In [1]:
import os
from pathlib import Path
import meeplemate
WORKSPACE_PATH = Path(meeplemate.__file__).parent.parent
os.chdir(WORKSPACE_PATH)

In [2]:
from IPython.display import display, Markdown
import importlib

from numpy import full
import dev_system
from meeplemate.config import GameService
importlib.reload(dev_system)
from langchain_openai import ChatOpenAI
from dev_system import areload, get_service
from meeplemate.component_system import factory

await areload(
    ["qa_service", "game_service", "chat_model", "chunk_search_service_2", "full_page_store", "tokenizer"],
)
qa_service = get_service("qa_service")
game_service: GameService = get_service("game_service")
chat_model = get_service("chat_model")
chunk_search_service = get_service("chunk_search_service_2")
full_page_store = get_service("full_page_store")
tokenizer = get_service("tokenizer")

Loading configuration from: config-dev.yaml
Loading configuration from: config-dev.yaml
Reloading system...
System reloaded


In [3]:
game_id = "warhammer_5th_edition"
manifest = await game_service.get_manifest(game_id)

## Queston Analysis

What is the user asking? What are the main rule interactions? What sort of question is it (e.g. rule lookup, rule interaction)?

In [4]:
from re import sub
from uuid import uuid4

from langchain_core.runnables import RunnableConfig
from meeplemate.qa_graph import (
    build_analyze_question_graph,
    QuestionAnalysisContext,
    create_question_analysis_state
)
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.globals import set_debug

set_debug(False)

agent = build_analyze_question_graph(
    InMemorySaver(),
    chat_model,
    tokenizer,
)

context = QuestionAnalysisContext(
    manifest=manifest,
    chunk_search_service=chunk_search_service,
)
input = create_question_analysis_state(
    # query="When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?"
    query="What is the leadership value of a Grail Knight",
    # query="Do Grail Knights have any special abilities that prevent them from taking Break tests?"
    # query="How does Grail Virtue interacts with Break test?"
)

config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

output = await agent.ainvoke(input, context=context, config=config)

display(Markdown(output["analysis"]))
print("\n\n")
print("Rule interactions:")
for interaction in output["rule_interactions"]:
    print(f"- {interaction}")

# print(output["reasoning"])
# # from pprint import pprint
# # pprint(output)
# # Display as Markdown
# display(Markdown(output["analysis"]["analysis"]))

# for subquestion in [subproblem["question"] for subproblem in output["analysis"]["subproblems"]]:
#     print(subquestion)

2026-02-07 06:05:53 [info     ] Tokens used by prompt          message_count=0 tokens_used=2998


2026-02-07 06:05:53 [info     ] search_chunks called           search_terms=['What is the leadership value of a Grail Knight'] token_budget=14978
2026-02-07 06:05:54 [info     ] Used tokens                    query=['What is the leadership value of a Grail Knight'] tokens_used=2921 total_retrieved_count=7
2026-02-07 06:05:54 [info     ] Retrieved results              query='What is the leadership value of a Grail Knight' relevant_count=7 total_retrieved_count=7
2026-02-07 06:05:54 [info     ] Tokens used by prompt          message_count=2 tokens_used=6365


/workspace/meeplemate/cassandra_util.py:39: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  return load(value) if value is not None else None


2026-02-07 06:05:57 [warning  ] No tool calls found, defaulting to extract_analysis
2026-02-07 06:05:59 [info     ] Determined direct rule interactions interactions=['Grail Knight profile interacts with Leadership value', 'Grail Knight special rule (Grail Virtue) interacts with psychology rules', 'Grail Knight profile interacts with knightly virtues', 'Grail Knight unit composition interacts with army list restrictions', 'Grail Knight stat line interacts with movement and combat mechanics']


The user is asking for the leadership value (Ld) of a Grail Knight, which is a specific unit type within the Bretonnia Army Book. This is a straightforward factual query about a game statistic.

The relevant rule involved in this question is the **Grail Knight profile**, which includes its leadership value. The documents provided contain multiple entries that explicitly list the stats for a Grail Knight, including the leadership value.

Relevant rules and mechanics:
1. **Grail Knight profile** — Contains the stat line including Leadership (Ld).
2. **Special Rules for Grail Knights** — Includes the "Grail Virtue" special rule, which explains immunity to psychology effects, but does not affect the leadership value directly.
3. **General table structure** — The inclusion of the Ld column in the stat line confirms that leadership is a defined attribute for units.

The key source passage clearly states the leadership value:

```
<table>
<tr><td></td><td>M</td><td>WS</td><td>BS</td><td>S</td><td>T</td><td>W</td><td>I</td><td>A</td><td>Ld</td></tr>
<tr><td>Grail Knight</td><td>4</td><td>5</td><td>3</td><td>4</td><td>3</td><td>1</td><td>4</td><td>1</td><td>9</td></tr>
</table>
```

This is the only rule necessary to answer the question.




Rule interactions:
- Grail Knight profile interacts with Leadership value
- Grail Knight special rule (Grail Virtue) interacts with psychology rules
- Grail Knight profile interacts with knightly virtues
- Grail Knight unit composition interacts with army list restrictions
- Grail Knight stat line interacts with movement and combat mechanics


In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from meeplemate.qa_graph import (
    build_question_answer_graph,
    build_coordinating_agent_graph,
    build_analyze_question_graph
)

analyze_graph = build_analyze_question_graph(
    InMemorySaver(),
    chat_model,
    tokenizer,
)

qa_graph = build_question_answer_graph(
    checkpoint_saver=InMemorySaver(),
    chat_model=chat_model,
    tokenizer=tokenizer,
)

coord_graph = build_coordinating_agent_graph(
    checkpoint_saver=InMemorySaver(),
    analyze_question_agent=analyze_graph,
    game_agent=qa_graph,
)

In [12]:
from typing import cast
from meeplemate.ingest.gamepackage import Manifest
from meeplemate.qa_graph import GameAgentContext, CoordinationInputState
from langchain_core.runnables import RunnableConfig
from uuid import uuid4

input: CoordinationInputState = {
    "query": "When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?",
}

context: GameAgentContext = GameAgentContext(
    manifest=cast(Manifest, manifest),
    chunk_search_service=chunk_search_service,
    full_page_store=full_page_store
)
config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

result = await coord_graph.ainvoke(input, context=context, config=config)

display(Markdown(result["response"]))

for item in result["evidence"]:
    display(Markdown(item["content"]))

2026-02-07 06:50:14 [info     ] Analyzing question             query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?'
2026-02-07 06:50:14 [info     ] Tokens used by prompt          message_count=0 tokens_used=3012
2026-02-07 06:50:14 [info     ] search_chunks called           search_terms=['What is a Break Test?', 'When do units take a Break Test?', 'Combat resolution mechanics', 'Grail Knights unit ability', 'Green Dragon unit ability'] token_budget=14964
2026-02-07 06:50:16 [info     ] Used tokens                    query=['What is a Break Test?', 'When do units take a Break Test?', 'Combat resolution mechanics', 'Grail Knights unit ability', 'Green Dragon unit ability'] tokens_used=11000 total_retrieved_count=28
2026-02-07 06:50:16 [info     ] Retrieved results              query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?' relevant_count=28 total_r

**Yes, Grail Knights must take a Break Test when they lose combat against a Green Dragon.**

The general rule for losing combat requires a Break Test:

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away. This is called a Break test. You need to take a separate Break test for every unit involved in the combat.

> (Warhammer Rulebook, p. 42)

This rule applies universally to all units that lose a combat, regardless of enemy type or special abilities.

However, the Green Dragon has a special ability that triggers a Leadership test due to corrosive fumes:

> Green Dragons belch corrosive green fumes. These acid clouds dissolve skin and irritate eyes. Any model hit suffers a Strength 4 hit with no saving throw for armour. In addition, a unit attacked by corrosive fumes may be forced to give ground before the choking clouds. The unit takes a Leadership test in the same way as for a fear or other psychology test (2D6 against its Leadership characteristic - see the Psychology section of the main rulebook for details).

> (Warhammer Battle Book, p. 126)

Despite this, the Grail Knights are protected from psychological effects:

> Grail Knights have the most noble chivalric virtue of all — the Grail Virtue. This means that they are unaffected by any of the psychology rules; any such tests they are called upon to take are disregarded with a cool and steely countenance. The Knight knows neither fear nor terror, nor will he panic, for the grail sustains his noble will better than any magic trickery.

> (Bretonnia Army Book, p. 44)

Further confirmation states:

> The unit never needs test for any of the psychology rules, whether panic, fear, terror or whatever. The Knights are unaffected by any psychology.

> (Bretonnia Army Book, p. 49)

This protection specifically prevents them from taking Leadership tests triggered by Psychology rules—such as those caused by the Green Dragon’s fumes.

However, the key distinction lies in the rulebook itself:

> However, a Break test is not a psychology test. The two tests are quite separate.

> (Warhammer Rulebook, p. 47)

This establishes that Break Tests and Psychology tests are functionally distinct. Therefore, even though the Grail Knights are immune to Psychology tests, they are not automatically exempt from Break Tests.

Since losing combat directly triggers a Break Test—and because the Break Test is not a Psychology test—the Grail Knights must still take a Break Test when they lose combat, regardless of the enemy's special abilities.

Thus, the Green Dragon’s fumes may prevent other units from surviving the combat through a Leadership test, but the Grail Knights are not affected by that effect. Nevertheless, they still face the mandatory Break Test due to losing the engagement.

Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.

## LOSERS TAKE A BREAK TEST

The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test. You need to take a separate Break test for every unit involved in the combat. Depending on which units pass and which fail their test, some may break and flee whilst others stand their ground. Troops which are better led, braver, and more professional are more likely to stand firm, whilst wild, temperamental troops are far more likely to run for it.

Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.

...

## TAKING PSYCHOLOGY TESTS

When taking psychology tests roll 2D6 and compare the result to your Leadership (Ld) value. If the result is less than or equal to the unit's Leadership score the test is passed and all is well. If the result is greater than the unit's Leadership score then the test is failed.

## PANIC TESTS FOR BREAKS

## LOSERS TAKE A BREAK TEST

## OTHER PSYCHOLOGY

Once they are within their own charge distance of enemy models frenzied units are not affected by other psychology. So long as they are within charge distance of the enemy they are immune to panic, fear, terror etc, and do not have to make these tests. Note that this immunity only extends to psychology tests, it does not include Break tests in hand- to- hand combat which must still be taken as normal.

Take the test as follows. Firstly, nominate which unit you are testing for. Roll 2D6 and add the scores together. Add the difference between the winner's and loser's combat score. If the total is greater than the unit's Leadership (Ld) value then the unit is broken. Broken units will turn tail and flee once all combat on the entire battlefield has been worked out. Until all combat has been worked out simply turn a few of the rear rank models round to remind you that the unit is broken.

In [11]:
from meeplemate.qa_graph import dump_chunks

for question in result["clarifying_questions"]:
    print("Clarifying question:", question["question"])
    # Display answer in markdown
    display(Markdown(question["answer"]))
    for item in question["evidence"]:
        display(Markdown(item["content"]))
    print("\n\n")

Clarifying question: How does Break test interacts with losing combat?


**The Break test is directly triggered by a unit losing combat and is influenced by the margin of that loss.**

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test.
>
> Take the test as follows. Firstly, nominate which unit you are testing for. Roll 2D6 and add the scores together. Add the difference between the winner's and loser's combat score. If the total is greater than the unit's Leadership (Ld) value then the unit is broken.

(Warhammer Rulebook, p. 42)

This rule establishes that losing a combat is the sole condition for triggering a Break test. The test itself uses the difference in combat scores — derived from the outcome of the combat — as a direct modifier to the dice roll. The combat score difference is added to the 2D6 roll, meaning the magnitude of the loss influences the likelihood of breaking.

> For example: A unit of Elf archers is fighting a unit of Goblin spearmen. The Goblin is fighting 3 wounds on the Elves, and the Elves inflict 4 wounds on the Goblin. However, the Goblin player has 4 complete ranks in his formation, and as each extra rank adds +1 to his score this gives him 3 + 3 = 6 points against the Elves' 4. The Elves have therefore lost the combat even though they have caused more casualties — the vast numbers of Goblin pressing from the back have overwhelmed them. The Elves must therefore take a Break test adding +2 to their dice score.

(Warhammer Rulebook, p. 42)

This example confirms that losing combat directly causes the need for a Break test, and that the combat’s own score difference (here, +2 due to rank bonuses) is explicitly applied as a modifier during the test.

Therefore, the interaction is both causal and quantitative: **losing combat triggers the Break test, and the combat’s outcome (specifically, the score difference) determines how difficult the test is.**

Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.

For example: A unit of Elf archers is fighting a unit of Goblin spearmen. The Goblin is fighting 3 wounds on the Elves, and the Elves inflict 4 wounds on the Goblin. However, the Goblin player has 4 complete ranks in his formation, and as each extra rank adds +1 to his score this gives him \(3 + 3 = 6\) points against the Elves' 4 The Elves have therefore lost the combat even though they have caused more casualties - the vast numbers of Goblin pressing from the back have overwhelmed them. The Elves must therefore take a Break test adding +2 to their dice score. Elves have a good Leadership value (8) but with the extra +2 modifier on the dice the player will have to roll 6 or less to stand and fight. The player rolls 2D6 and scores 7, the +2 modifier brings his total to 9 which is greater than the unit's Leadership so the Elves are broken.

Take the test as follows. Firstly, nominate which unit you are testing for. Roll 2D6 and add the scores together. Add the difference between the winner's and loser's combat score. If the total is greater than the unit's Leadership (Ld) value then the unit is broken. Broken units will turn tail and flee once all combat on the entire battlefield has been worked out. Until all combat has been worked out simply turn a few of the rear rank models round to remind you that the unit is broken.

...

"LOSERS TAKE A BREAK TEST" (p. 42)

## TAKING PSYCHOLOGY TESTS

When taking psychology tests roll 2D6 and compare the result to your Leadership (Ld) value. If the result is less than or equal to the unit's Leadership score the test is passed and all is well. If the result is greater than the unit's Leadership score then the test is failed.




Clarifying question: How does Fear test interacts with Break test?


**No, a Fear test does not count as a Break test.**

Fear tests and Break tests are distinct mechanics with separate triggers and outcomes.

> A unit must take a Fear test in the following situations:  
> 1. If Charged by a Feared Enemy.  
> 2. If a Unit Wishes to Charge a Feared Enemy.  
> 3. Defeated by Feared Enemy — Automatic Break without Break test if outnumbered.

(Warhammer Rulebook, p. 49)

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away. This is called a Break test. Take the test as follows. Firstly, nominate which unit you are testing for. Roll 2D6 and add the scores together. Add the difference between the winner's and loser's combat score. If the total is greater than the unit's Leadership (Ld) value then the unit is broken.

(Warhammer Rulebook, p. 42)

> Note that a battle standard allows a unit to retake a failed Break test — and only a Break test. A battle standard does not entitle a unit to retake any other Leadership test, such as a psychology test or a test to rally.

(Warhammer Rulebook, p. 89)

This confirms that Break tests and psychology tests—such as Fear tests—are treated as separate categories. The rule explicitly states that a battle standard only allows a retake of a **Break test**, not any other Leadership test, meaning the two are functionally distinct.

Furthermore, there is one specific case where a Fear condition overrides a Break test:

> A unit defeated in hand-to-hand combat is automatically broken without a Break test if it is fighting an enemy that it fears and which outnumbers it.

(Warhammer Rulebook, p. 49)

This exception shows that while Fear and Break tests are not the same, they can interact under specific conditions—but only when both fear and outnumbering are true. In all other cases, the two remain entirely independent.

Therefore, a Fear test is **not** equivalent to a Break test. They are different types of tests with different rules, triggers, and effects. The presence of a Fear test does not substitute for or affect a Break test unless specifically overridden by the stated exception.

## DEFEATED BY FEARED ENEMY

A unit defeated in hand- to- hand combat is automatically broken without a Break test if it is fighting an enemy that it fears and which outnumbers it. If the fear- causing enemy does not outnumber the unit then a Break test is taken as normal. See the Close Combat section for details of combat results, Break tests and fleeing troops.

In [7]:
from uuid import uuid4
from langchain_core.globals import set_debug

# set_debug(True)

input = {
    "manifest": manifest,
    "query": "How does Grail Virtue interact Break test?",
    "recursion_depth": 0,
    "evidence": [],
    "messages": [],
}

results = []
for _ in range(10):
    responses = await qa_service.abatch([input]*10, config={"configurable": {"thread_id": str(uuid4())}})

    for response in responses:
        result = response["response"].splitlines()[0]
        results.append(result)
        print(result)
        print()

# # Display response as markdown
# from IPython.display import Markdown, display
# display(Markdown(response["response"]))

2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Retrieving data for query      query='How does Grail Virtue interact Break test?'
2026-02-04 19:39:24 [info     ] Retrieving data for query      query='How does Grail Virtue interact Break test?'
2026-02-04 19:39:24 [info     ] Retrieving data for query      query='How does Grail Virtue interact Break test?'
2026-02-04 19:39:24 [info   

CancelledError: 

In [ ]:
# Right 95% of the time.
for result in sorted(results):
    print(result)

**Grail Knights are not exempt from Break tests, despite their Grail Virtue.**
**No, Grail Knights are not exempt from Break tests, even though they are unaffected by psychology rules.**
**No, Grail Knights are not exempt from Break tests, even though they have the Grail Virtue.**
**No, Grail Knights are not exempt from Break tests.**
**No, Grail Knights are not exempt from Break tests.**
**No, Grail Knights are not protected from Break tests by the Grail Virtue.**
**No, Grail Knights are not protected from Break tests by the Grail Virtue.**
**No, Grail Knights do not automatically avoid Break tests, even though they are unaffected by psychology.**
**No, Grail Knights do not bypass Break tests due to Grail Virtue.**
**No, Grail Knights do not bypass Break tests, even though they are granted immunity to psychology.**
**No, Grail Knights do not bypass Break tests, even though they possess the Grail Virtue.**
**No, Grail Knights do not gain immunity to Break tests from the Grail Virtue.**

In [13]:
for chunk in response["evidence"]:
    header = f"### From {chunk['rulebook_name']} page {chunk['page']}\n"
    display(Markdown(header + chunk["content"]))


### From Warhammer Rulebook page 47
Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.